# Christopher: vanilla cofactor order

In [2]:
import csv
import ast
import pandas as pd
import matplotlib.pyplot as plt

def csv2dict(csv_file_path):
    result_dict = {}
    with open(csv_file_path, 'r') as csv_file:
        csv_reader = csv.reader(csv_file)
        
        for row in csv_reader:
            try:
                result_dict[row[0]] = ast.literal_eval(row[1])  # value is list
            except:
                result_dict[row[0]] = row[1]
    return result_dict

def dict2csv(my_dict, csv_file_path):
    with open(csv_file_path, 'w', newline='') as csv_file:
        csv_writer = csv.writer(csv_file)
        for key, value in my_dict.items():
            csv_writer.writerow([key] + [value])

In [6]:
cpd2name = csv2dict('../data/assets/cpd2nameShort.csv')
rn2cpds = csv2dict('../data/assets/rn2cpds_SI.csv')

In [4]:
import networkExpansionPy.folds as nf
import pandas as pd
from collections import Counter
from pathlib import PurePath, Path

# seed = sys.argv[1]
# random.seed(seed)
asset_path = nf.asset_path

# for vanilla
METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.01May2023.pkl") # path to metabolism object pickle
RN2RULES_PATH = PurePath(asset_path,"rn2fold","rn2rules.20230224.pkl") # path to rn2rules object pickle
SEED_CPDS_PATH = PurePath(asset_path, "compounds", "seeds.Goldford2022.csv") # path to seed compounds csv

# for FOLD-GATED
ALGORITHM = "no_look_ahead_rules"
WRITE = True # write result to disk
WRITE_TMP = False # write after each iteration
CUSTOM_WRITE_PATH = None # if writing result, custom path to write to
STR_TO_APPEND_TO_FNAME = None # if writing result, str to append to filename
ORDERED_OUTCOME = False # ignore random seed and always choose folds based on sort order
IGNORE_REACTION_VERSIONS = True # when maximizing for reactions, don't count versioned reactions

## Metabolism
metabolism = pd.read_pickle(METABOLISM_PATH)
# remove reactions that produce H2O2 before O2
H2O2_rns = ['R00017_v1', 'R03532_v1', 'R09507_v1', 'R09740_v1', 'R09741_v1', 'R11522', 'R12455', 'R12454']
condition = ((metabolism.network['rn'].isin(H2O2_rns)) & (metabolism.network['direction'] == 'reverse'))
metabolism.network = metabolism.network[~condition]
assert 'R00017_v1' not in list(metabolism.network['rn'])

## Ablation
# metabolism.pruneReactionsFromMetabolite(['C00024'])

## FoldRules
rn2rules = pd.read_pickle(RN2RULES_PATH)
foldrules = nf.FoldRules.from_rn2rules(rn2rules)

## Modify seeds with AA and GATP_rns
aa_cids = set(["C00037",
    "C00041",
    "C00065",
    "C00188",
    "C00183",
    "C00407",
    "C00123",
    "C00148",
    "C00049",
    "C00025"])

GATP_rns = {'R00200_gATP_v1',
    'R00200_gATP_v2',
    'R00430_gGTP_v1',
    'R00430_gGTP_v2',
    'R01523_gATP_v1',
    'R04144_gATP_v1',
    'R04208_gATP',
    'R04463_gATP',
    'R04591_gATP_v1',
    'R06836_gATP',
    'R06974_gATP',
    'R06975_gATP_v1'}

## Seed
seed = nf.Params(
    rns = set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
    cpds = set((pd.read_csv(SEED_CPDS_PATH)["ID"])) | aa_cids,
    folds = set(['spontaneous'])
)

## Seed with pre-expansion
# preALL = pd.read_pickle('../data/pre-expansion_seed_cpds/pre-expansion_seed_cpds_ALL.pkl')
# seed = nf.Params(
#     rns = set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#     cpds = set(preALL) | aa_cids,
#     folds = set(['spontaneous'])
# )

## Inititalize fold metabolism
fm = nf.FoldMetabolism(metabolism, foldrules, seed)

calculating scope...


/Users/tseamuscorlett/Desktop/LongoLab/Fold_Gated/networkExpansion/networkExpansionPy/lib.py:556: SparseEfficiencyWarning: Comparing sparse matrices using == is inefficient, try using != instead.
  X,Y = netExp_trace(R,P,x0,b)


...done.


In [18]:
cofactors = ['C00018', 'Z00035', 'C00002', 'C00003', 'Z00032', 'C00019', 'Z00041', 'C00010', 'Z00051', 'C00061', 'Z00014', 'C00016', 'Z00013', 'C00068', 'Z00047', 'C00055', 'C00063', 'C00044', 'C00130', 'C00075', 'C00131', 'C00458', 'C00286', 'C00365', 'Z00009']

cpd2rns = {}
for c in cofactors:
    rns = []
    for rn, cpds in rn2cpds.items():
        if c in cpds:
            rns.append(rn)
    cpd2rns[c] = rns

In [20]:
data = []
for c in cofactors:
    data.append({
    'iteration': fm.scope.cpd_iteration_dict[c],
    'KEGG_ID': c,
    'name': cpd2name[c],
    'num_reactions': len(cpd2rns[c]),
    'reactions': cpd2rns[c]
    })

df = pd.DataFrame(data)
df
# df.to_csv('cofactor2iteration.csv')

In [ ]:
# Adenosine triphosphate / ATP (C00002)
# Cobalamin (B12)
# CoA + zCoA  (C00010 + Z00051)
# SAM + zSAM + SAH (C00019 + Z00041)
# ThDP_zThDP  (C00068 + Z00047)
# NAD_NADH_zNAD  ('C00003' + Z00032)
# FAD_FADH2_zFAD  (C00016 + Z00013)
# FMN_zFMN  (C00061 + Z00014)

# PLP_zPLP (C00018 + Z00035)

# Cytidine monophosphate / CMP (C00055)
# Cytidine triphosphate / CTP (C00063)
# Guanosine triphosphate / GTP (C00044)
# Inosine monophosphate / IMP (C00130)
# Thymidine triphosphate / TTP (not on KEGG)
# Uridine triphosphate / UTP (C00075)

# Deoxyadenosine triphosphate / dATP (C00131)
# Deoxycytidine triphosphate / dCTP (C00458)
# Deoxyguanosine triphosphate / dGTP (C00286)
# Deoxyuridine monophosphate / dUMP (C00365)

In [13]:
for c, name in cpd2name.items():
    if c.startswith('Z'):
        print(c, name)

Z00001 2Fe2S
Z00002 4Fe4S
Z00003 ACP
Z00004 Ascorbate
Z00005 Biotin
Z00006 Cobalt
Z00007 CoB
Z00008 CoM
Z00009 Cobalamin
Z00010 Cytochrome
Z00011 F420
Z00012 F430
Z00013 FAD
Z00014 FMN
Z00015 Iron
Z00016 Ferredoxin
Z00017 Flavin
Z00018 Flavodoxin
Z00019 Flavoprotein
Z00020 Generic FeS Cluster
Z00021 Generic Flavin
Z00022 Generic quinone
Z00023 Glutathione
Z00024 H4MPT
Z00025 Heme
Z00026 K (Potassium)
Z00027 Lipoic acid
Z00028 Methanofuran
Z00029 Mg
Z00030 Mn
Z00031 Molybdopterin
Z00032 NAD/NADP
Z00033 Sodium
Z00034 Nickel
Z00035 PLP
Z00036 PQQ
Z00037 Pantetheine
Z00038 Phosphopantetheine
Z00039 Pyruvoyl
Z00040 Riboflavin
Z00041 SAM
Z00042 Selenium
Z00043 Selenide
Z00044 Selenocysteine
Z00045 THB
Z00046 THF (H4F)
Z00047 Thiamine diphosphate
Z00048 Thiamine phosphate
Z00049 Ubiquinone
Z00050 Vitamin K
Z00051 Coenzyme A
Z00052 Glycyl radical
Z00053 Tungsten
Z00054 Zinc
Z00055 Calcium
Z00056 Dipyrromethane
Z00057 prenFMN
Z00058 Chlorophyll
Z00059 Tungstopterin
Z00060 Monovalent Metal (M1)
